In [1]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("superstore.csv", encoding="latin1")

# Display first few rows
print(df.head())

# Check column names and data types
print(df.info())


   Row ID        Order ID  Order Date   Ship Date       Ship Mode Customer ID  \
0       1  CA-2016-152156  2016-11-08  2016-11-11    Second Class    CG-12520   
1       2  CA-2016-152156  2016-11-08  2016-11-11    Second Class    CG-12520   
2       3  CA-2016-138688  2016-06-12  2016-06-16    Second Class    DV-13045   
3       4  US-2015-108966  2015-10-11  2015-10-18  Standard Class    SO-20335   
4       5  US-2015-108966  2015-10-11  2015-10-18  Standard Class    SO-20335   

     Customer Name    Segment        Country             City  ...  \
0      Claire Gute   Consumer  United States        Henderson  ...   
1      Claire Gute   Consumer  United States        Henderson  ...   
2  Darrin Van Huff  Corporate  United States      Los Angeles  ...   
3   Sean O'Donnell   Consumer  United States  Fort Lauderdale  ...   
4   Sean O'Donnell   Consumer  United States  Fort Lauderdale  ...   

  Postal Code  Region       Product ID         Category Sub-Category  \
0       42420   Sout

In [2]:
# ----- Basic info and cleaning -----
# 1) shape and columns
print("rows,cols:", df.shape)
print(df.columns.tolist())

# 2) rename columns for consistency (optional)
df.columns = [c.strip().replace(' ', '_') for c in df.columns]

# 3) check missing values
print(df.isnull().sum())

# 4) drop rows missing critical values (OrderID, Sales)
df = df.dropna(subset=['Order_ID','Sales'])

# 5) convert dates
df['Order_Date'] = pd.to_datetime(df['Order_Date'], errors='coerce')
df['Ship_Date'] = pd.to_datetime(df['Ship_Date'], errors='coerce')
# If date conversion created NaT, you migt drop or fill them:
df = df.dropna(subset=['Order_Date'])


rows,cols: (9994, 21)
['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']
Row_ID           0
Order_ID         0
Order_Date       0
Ship_Date        0
Ship_Mode        0
Customer_ID      0
Customer_Name    0
Segment          0
Country          0
City             0
State            0
Postal_Code      0
Region           0
Product_ID       0
Category         0
Sub-Category     0
Product_Name     0
Sales            0
Quantity         0
Discount         0
Profit           0
dtype: int64


In [3]:
# Derive a proxy for inventory days (simplified)
sales_per_day = df.groupby('Product_ID')['Quantity'].sum() / df['Order_Date'].nunique()
df['Inventory_Days'] = df['Quantity'] / (df['Product_ID'].map(sales_per_day) + 1e-5)


In [4]:
# Compute correlation
correlation = df[['Inventory_Days', 'Profit']].corr().iloc[0,1]
print(f"Correlation between Inventory Days and Profitability: {correlation:.2f}")


Correlation between Inventory Days and Profitability: 0.04


In [5]:
# Classify based on Inventory_Days
conditions = [
    (df['Inventory_Days'] > df['Inventory_Days'].quantile(0.75)),
    (df['Inventory_Days'] < df['Inventory_Days'].quantile(0.25))
]
choices = ['Overstocked', 'Fast-moving']
df['Stock_Status'] = np.select(conditions, choices, default='Normal')

# Average profit by stock type
stock_profit = df.groupby('Stock_Status')['Profit'].mean()
print(stock_profit)


Stock_Status
Fast-moving    18.324256
Normal         24.976443
Overstocked    46.297603
Name: Profit, dtype: float64
